# 12.b — Ad hoc test: maps of the changes beyond the internal variability, cell by cell

**Ad hoc test, outside the suitability chain**: it reads the delta BIO variables of the whole runs and of their two
halves written by 12.a (`test_internal_variability/<route>/<set>/bio/`, nothing is recomputed from the daily climate)
and writes only in `test_internal_variability/<route>/<set>/robust/`.

**Question.** 12.a showed that the totals of the effect of the AMOC weakening are robust, but that between the two
halves of one run about as many cells change class as between hosing and no hosing: on the change maps of 6.b about
half of the single switches could be year-to-year noise. Which cells change beyond the variability?

**Rule.** The final suitability of 4.g / 6.b (SVM with cross-fitting, novel climate, ECOCROP TMIN on the lower part
of the cell, soil ≥ 10) of each run in three parts: the whole run (full, 49 seasons) and its halves H1 (24) and H2
(25). For each pair (hosing − no hosing, same CO₂ level) and part, a cell assessable in both runs is gained (+1), lost
(−1) or unchanged (0). A cell is a **robust gain / loss** when it is gained / lost in the whole run *and* in each half
(three estimates, the two halves independent); an **uncertain change** when it changes in the whole run but not the
same way in both halves; changes seen only in a half are noise. Like the hatching of climate projections where the
signal exceeds the internal variability (Tebaldi et al. 2011, *Geophys. Res. Lett.* 38, L23701), but on a binary
class; conservative (a real change of a cell close to the threshold may miss it in one half).

**Outputs.** Per crop and input level: `robust_changes_<crop>.nc` (class per pair: 0 suitable in neither, 1 in both, 2
robust loss, 3 robust gain, 4 uncertain loss, 5 uncertain gain, 6 novel in either; full suitability of both runs),
`robust_areas_<set>.csv` (land in Mha and share of today's crop area), maps of 2050 for every crop (high input).

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: as 12.a
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]
KERNEL_EPS = 0.01
N_JOBS = min(5, os.cpu_count() or 1)
PARTS = ["full", "H1", "H2"]
MAP_LEVEL = "high"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
V12 = cu.out_dir(CFG, "test_internal_variability", ROUTE, SET)
BIO_OUT = V12 / "bio"                                                    # delta BIO of 12.a (read only)
OUT = V12 / "robust"; OUT.mkdir(exist_ok=True)                          # the only folder written

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
DT = cu.tmin_warming(CFG)
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
half_file = lambda run, part, crop: BIO_OUT / f"BIOvariables_delta_{run}_{part}_{crop}.nc"
missing = [half_file(r, p, c).name for r in RUNS for p in PARTS for c in CROPS if not half_file(r, p, c).exists()]
assert not missing, f"run 12.a first: {missing[:3]} ..."
with xr.open_dataset(era5_file(CROPS[0])) as d0:
    COLS = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e") and v not in ("BIO03", "BIO08")]
print(f"set {SET} | delta BIO of 12.a in {BIO_OUT} | written only in {OUT}")

## 1. Final suitability of every run and part (the function of 12.a)

In [ ]:
def suitability(crop):
    # {(run, part): {"suit": {level: bool (lat, lon)}, "novel": bool}}, as 4.g + 6.b (12.a without ERA5)
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    assert set(cols) <= set(COLS), (cols, COLS)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        REF = cu.bio_array(ds, cols, CFG)[ii, jj]
    REF = REF[np.isfinite(REF).all(axis=1)]
    kfun = cu.kernel_support(bundle["model"])
    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                            tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    ssr = {lv: np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0) for lv in LEVELS}
    which = cu.ECOCROP_TMIN[crop][1]
    out = {}
    for key, path in {(r, p): half_file(r, p, crop) for r in RUNS for p in PARTS}.items():
        with xr.open_dataset(path) as ds:
            A = cu.bio_array(ds, cols, CFG)
            tg = np.asarray(cu.growing_temperature(ds, which), float)
        X = A.reshape(-1, len(cols))
        ok = np.isfinite(X).all(axis=1) & LAND.ravel()
        p, nov = np.full(X.shape[0], np.nan), np.zeros(X.shape[0], bool)
        if ok.any():
            Xs = bundle["scaler"].transform(X[ok])
            p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
            nov[ok] = (cu.mess(X[ok], REF)[0] < 0) | (cu.nt2(X[ok], REF) > 1) | (kfun(Xs) < KERNEL_EPS)
        p, nov = p.reshape(LAND.shape), nov.reshape(LAND.shape)
        g = cu.crossfit_proba(Fd, A[ii, jj])
        fin = np.isfinite(g)
        p[ii[fin], jj[fin]] = g[fin]
        p = cu.balanced_proba(p, n1, n0)
        nov &= np.isfinite(p)
        proba = np.where(nov, 0.0, p)
        clim = cu.climate_suitable(proba, tg, crop, DT) & np.isfinite(proba)
        out[key] = {"suit": {lv: LAND & cu.suitable(clim, ssr[lv]) for lv in LEVELS}, "novel": nov}
    return out


S = {}
for crop in CROPS:
    t0 = time.time()
    S[crop] = suitability(crop)
    print(f"{crop}: {len(S[crop])} climates in {(time.time() - t0) / 60:.1f} min", flush=True)

## 2. Classes, areas and maps

Classes per cell: 0 suitable in neither run, 1 in both, 2 robust loss, 3 robust gain, 4 uncertain loss, 5 uncertain
gain (changes in the whole run, not the same way in both halves), 6 novel climate in either run (whole run). Areas:
land (Mha) and share of today's crop area (MIRCA2000) per class, with the net robust change of today's crop area.

In [ ]:
mha = lambda m: float(np.nansum(LAND_MHA * m))
NAMES = ["neither", "both", "robust loss", "robust gain", "uncertain loss", "uncertain gain", "novel in either"]


def classes(R, h, o, lv):
    full = R[(h, "full")]["suit"][lv].astype(int) - R[(o, "full")]["suit"][lv].astype(int)
    ch = {p: R[(h, p)]["suit"][lv].astype(int) - R[(o, p)]["suit"][lv].astype(int) for p in ("H1", "H2")}
    both = R[(h, "full")]["suit"][lv] & R[(o, "full")]["suit"][lv]
    c = np.where(both, 1, 0)
    agree = lambda v: (full == v) & (ch["H1"] == v) & (ch["H2"] == v)
    c = np.where(full == -1, np.where(agree(-1), 2, 4), c)
    c = np.where(full == 1, np.where(agree(1), 3, 5), c)
    nov = R[(h, "full")]["novel"] | R[(o, "full")]["novel"]
    return np.where(LAND, np.where(nov, 6, c), -1)


rows, CL = [], {}
for crop in CROPS:
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    dv = {}
    for lv in LEVELS:
        for h, o in PAIRS:
            c = classes(S[crop], h, o, lv)
            CL[(crop, lv, h)] = c
            yr = h.split("_")[1] if "_" in h else h
            dv[f"class_{yr}_{lv}"] = (("lat", "lon"), c.astype("int8"))
            r = {"crop": crop, "level": lv, "pair": f"{h} - {o}"}
            for k, n in enumerate(NAMES):
                if k in (2, 3, 4, 5):
                    r[f"{n} (Mha)"] = mha(c == k)
                    r[f"{n}, today's crop area (%)"] = share(c == k)
            r["net change of today's crop area, all (points)"] = share((c == 3) | (c == 5)) - share((c == 2) | (c == 4))
            r["net change of today's crop area, robust only (points)"] = share(c == 3) - share(c == 2)
            ch = r["robust loss, today's crop area (%)"] + r["robust gain, today's crop area (%)"]
            r["robust share of the switches of today's crop area (%)"] = 100 * ch / max(ch + r["uncertain loss, today's crop area (%)"] + r["uncertain gain, today's crop area (%)"], 1e-9)
            rows.append(r)
    xr.Dataset(dv, coords={"lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "classes": ", ".join(f"{k} {n}" for k, n in enumerate(NAMES)), "set": SET,
                      "rule": "robust = the same change in the whole run and in both halves (12.a)"}).to_netcdf(OUT / f"robust_changes_{crop}.nc")
AREAS = pd.DataFrame(rows).set_index(["crop", "level", "pair"])
AREAS.to_csv(OUT / f"robust_areas_{SET}.csv")
with pd.option_context("display.width", 250, "display.max_columns", 30):
    display(AREAS.xs(MAP_LEVEL, level="level").round(2))

COL = ["#e6e6e6", "#c7e9c0", "#d7191c", "#1a9641", "#fdae61", "#a6d96a", "#9e9e9e"]
LAB = ["not suitable in both", "suitable in both", "lost, robust", "gained, robust", "lost, uncertain (not in both halves)",
       "gained, uncertain (not in both halves)", "climate never seen in training"]
h2050 = PAIRS[-1][0]
fig, axes = plt.subplots((len(CROPS) + 1) // 2, 2, figsize=(26, 5.6 * ((len(CROPS) + 1) // 2)), squeeze=False)
for ax, crop in zip(axes.ravel(), CROPS):
    c = CL[(crop, MAP_LEVEL, h2050)]
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(c >= 0, c, np.nan), cmap=ListedColormap(COL), vmin=-0.5, vmax=6.5, shading="nearest")
    a = AREAS.loc[(crop, MAP_LEVEL, f"{h2050} - {PAIRS[-1][1]}")]
    ax.set_ylim(-60, 85)
    ax.set_title(f"{crop}, {MAP_LEVEL} input, 2050 hosing vs no hosing: robust gain {a['robust gain (Mha)']:.0f} / loss "
                 f"{a['robust loss (Mha)']:.0f} Mha, uncertain gain {a['uncertain gain (Mha)']:.0f} / loss {a['uncertain loss (Mha)']:.0f} Mha", fontsize=10)
for ax in axes.ravel()[len(CROPS):]:
    ax.axis("off")
axes.ravel()[-1].legend(handles=[Patch(color=c_, label=l) for c_, l in zip(COL, LAB)], loc="center", fontsize=12)
fig.tight_layout(); fig.savefig(OUT / f"robust_changes_2050_{MAP_LEVEL}.png", dpi=90, bbox_inches="tight"); plt.show()